In [2]:
!pip install openpyxl

   ---------------------------------------- 0.0/250.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/250.9 kB ? eta -:--:--
   - -------------------------------------- 10.2/250.9 kB ? eta -:--:--
   ------ -------------------------------- 41.0/250.9 kB 495.5 kB/s eta 0:00:01
   ----------------- -------------------- 112.6/250.9 kB 939.4 kB/s eta 0:00:01
   ---------------------------------------  245.8/250.9 kB 1.5 MB/s eta 0:00:01
   ---------------------------------------- 250.9/250.9 kB 1.4 MB/s eta 0:00:00



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: C:\Users\Evander\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [7]:
import pandas as pd
import glob
import os

input_folder = '../datasets/raw_datasets/pihps_unmerged/'
output_folder = '../datasets/raw_datasets/'

search_path = os.path.join(input_folder, 'Idul Adha *.xlsx')
files = glob.glob(search_path)

print(f"Menemukan {len(files)} file Excel PIHPS untuk diproses.")

df_list = []

for f in files:
    df = pd.read_excel(f)
    nama_tahun = os.path.basename(f).replace('.xlsx', '')
    df.columns.values[0] = 'No'
    df.columns.values[1] = 'Komoditas'
    df = df.dropna(subset=['Komoditas'])
    df_melted = df.melt(
        id_vars=['No', 'Komoditas'], 
        var_name='Tanggal_Raw', 
        value_name='Harga'
    )
    df_melted['Tahun_Sumber'] = nama_tahun
    df_list.append(df_melted)

df_total_harga = pd.concat(df_list, ignore_index=True)
df_total_harga['Tanggal_Raw'] = df_total_harga['Tanggal_Raw'].astype(str).str.replace(' ', '')
df_total_harga = df_total_harga[df_total_harga['Harga'].astype(str).str.strip() != '-']
df_total_harga = df_total_harga.dropna(subset=['Harga'])
df_total_harga['Harga'] = df_total_harga['Harga'].astype(str).str.replace(',', '').str.replace('.', '')
df_total_harga['Harga'] = pd.to_numeric(df_total_harga['Harga'], errors='coerce')
output_file = os.path.join(output_folder, 'all_price_iduladha_2017_2026.csv')
df_total_harga.to_csv(output_file, index=False)

print(f"\nFile gabungan disimpan di: {output_file}")
print(f"Total baris data terkumpul: {len(df_total_harga)} baris.")

Menemukan 10 file Excel PIHPS untuk diproses.

File gabungan disimpan di: ../datasets/raw_datasets/all_price_iduladha_2017_2026.csv
Total baris data terkumpul: 9455 baris.


In [10]:
input_file = '../datasets/raw_datasets/weather_nasa_untransformed/POWER_Point_Daily_20170101_20260527_006d92S_107d61E_LST.csv'
output_folder = '../datasets/raw_datasets/'

df_weather = pd.read_csv(input_file, skiprows=17)
print("Kolom asli:", df_weather.columns.tolist())

df_weather['Date'] = pd.to_datetime(
    df_weather['YEAR'].astype(str) + ' ' + df_weather['DOY'].astype(str), 
    format='%Y %j'
)
df_weather['Date'] = df_weather['Date'].dt.strftime('%Y-%m-%d')
kolom_cuaca = ['Date'] + [col for col in df_weather.columns if col not in ['Date', 'YEAR', 'DOY']]
df_weather_clean = df_weather[kolom_cuaca]
output_file = os.path.join(output_folder, 'all_weather_nasa_2017_2026.csv')
df_weather_clean.to_csv(output_file, index=False)

print(f"\nData terformat (penyesuaian tanggal) disimpan di: {output_file}")
print(f"Total baris data: {len(df_weather_clean)} baris.")
print(df_weather_clean.head(3))

Kolom asli: ['YEAR', 'DOY', 'T2M_MIN', 'T2M_MAX', 'T2M', 'RH2M', 'PRECTOTCORR', 'ALLSKY_SFC_SW_DWN', 'WS10M_MAX', 'WS10M', 'WD10M']

Data terformat (penyesuaian tanggal) disimpan di: ../datasets/raw_datasets/all_weather_nasa_2017_2026.csv
Total baris data: 3434 baris.
         Date  T2M_MIN  T2M_MAX    T2M   RH2M  PRECTOTCORR  ALLSKY_SFC_SW_DWN  \
0  2017-01-01    18.39    26.51  22.04  87.64         3.50              21.98   
1  2017-01-02    18.39    25.46  21.50  88.15         2.86              23.36   
2  2017-01-03    17.93    26.56  22.05  87.92         3.18              19.35   

   WS10M_MAX  WS10M  WD10M  
0       3.73   2.58  240.2  
1       5.31   3.40  232.6  
2       4.77   3.12  251.8  
